# 05 · Road teacher (trajectory-based LiDAR-camera road autolabels)

Soft free-road labels for the **training** drives, without any human labels (method of Alamikkotervo et al., IEEE Access 2025; re-implemented in `ocgf/teachers/road/`, MIT notice kept, no CRF).

Per frame: the next 50 m of the ego path (SLAM poses) gives a trajectory point per LiDAR ring and a wheel point either side; LiDAR points are scored by their height and height gradient relative to those wheel points; the scores are interpolated into the image (LiDAR label). DINOv2 ViT-g/14 features of the path pixels give a road prototype, and cosine similarity to it gives the camera label. The label is their mean, in [0, 1]; NaN = unknown. A frame whose path finds no trajectory stays all unknown.

It needs future poses, so it is a **training-time teacher only**: the student never sees it at inference, and nothing is ever evaluated against it. Frames with KITTI-360 2D semantics are *audited* against them (road IoU, reported only, never used to tune).

Frames: the usable frames sampled every 5 m.

## Before you run

1. **Settings → Accelerator:** GPU T4 (one is enough). **Internet:** On (code clone and the DINOv2 weights, about 4 GB).
2. **Add Input:**
   - drive 0000: the output of the data-prep notebook for drive 0000, the KITTI-360 starter (images, calibration, poses);
   - other drives: their `kitti360-d<drive>-supplement`, the starter (calibration) and `data-poses`;
   - optional, for the audit: the 2D semantics (`kitti360-2d-semantics`).
3. **Save Version → Save & Run All.** Drives without attached inputs are skipped and listed.

In [ ]:
from pathlib import Path

REPO_URL  = "https://github.com/GNaveenkumar614/OCGF.git"
GIT_REF   = "p0/skeleton-geometry"       # branch, tag or commit (main once merged); pin a commit for the final run
DRIVES    = ["0000"]                     # then the training drives of kitti360_paper_v1: "0002", "0004", "0005"
INPUT_ROOT = Path("/kaggle/input")
OUT_DIR   = Path("/kaggle/working/road_labels")  # kept as the notebook output
STAGE_DIR = Path("/kaggle/temp/road_inputs")     # local copy of the frames' scans, images and 2D semantics
THREADS   = 32                                   # parallel readers: /kaggle/input is slow per file
CAMERA    = "dinov2_vitg14_reg"                  # "none" runs the LiDAR step only
QA        = 8                                    # QA figures per drive
SAVE_PARTS = True                                # also keep the LiDAR and camera labels (for the audit and ablations)
MAX_FRAMES = None                                # e.g. 50 for a trial
ALL_FRAMES = False                               # True: every stored usable frame, not one every 5 m
CODE_DIR  = Path("/kaggle/temp/OCGF")

## Code

A fresh clone of the repo at `GIT_REF`, installed in editable mode.

In [ ]:
import os, shutil, subprocess, sys

def sh(cmd, cwd=None):
    # run a command, stream its output into this notebook, and stop Run All if it fails
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")

CODE_DIR.parent.mkdir(parents=True, exist_ok=True)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)            # fresh clone, so the run matches GIT_REF exactly
sh(f'git clone -q {REPO_URL} "{CODE_DIR}"')
sh(f'git -C "{CODE_DIR}" checkout -q {GIT_REF}')
sh(f'git -C "{CODE_DIR}" log -1 --date=short --format="code at commit %h (%cd): %s"')
sh(f'{sys.executable} -m pip install -q -e "{CODE_DIR}"')
sys.path.insert(0, str(CODE_DIR))
for name in [m for m in sys.modules if m == "ocgf" or m.startswith("ocgf.")]:
    del sys.modules[name]

from ocgf.utils.provenance import environment
print(environment())
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

## Inputs

Each drive needs its data-prep output and the raw inputs it points to. The 2D semantics are optional (audit only).

In [ ]:
from ocgf.datasets.locate import describe_prepared, prepared_drive

ready = []
for d in DRIVES:
    seq = f"2013_05_28_drive_{d}_sync"
    try:
        parts = prepared_drive(INPUT_ROOT, seq)
    except FileNotFoundError as err:
        print(f"drive {d}: skipped, {err}")
        continue
    audit = f"audit against {parts['semantics2d']}" if "semantics2d" in parts else "no 2D semantics attached, no audit"
    print(describe_prepared(INPUT_ROOT, seq))
    print(f"drive {d}: ready; {audit}")
    ready.append(seq)
assert ready, "no drive is ready: attach the drive inputs (Add Input)"


## Label

One drive at a time: the selected frames are copied to local disk first, then labelled in driving order (the camera step reuses the previous frame's road prototype when a path is too short).

In [ ]:
for seq in ready:
    sh(f'{sys.executable} scripts/run_road_teacher.py --input-root "{INPUT_ROOT}" --sequence {seq} --out "{OUT_DIR}" '
       f'--camera {CAMERA} --qa {QA} --stage-dir "{STAGE_DIR}" --threads {THREADS}'
       + (" --save-parts" if SAVE_PARTS else "") + (" --all-frames" if ALL_FRAMES else "")
       + (f" --max-frames {MAX_FRAMES}" if MAX_FRAMES else ""), cwd=CODE_DIR)
    shutil.rmtree(STAGE_DIR / seq, ignore_errors=True)     # free the local disk for the next drive

## Summary and QA figures

Frame statuses and the audit against the KITTI-360 2D road (label 7) per drive: the combined label, and the LiDAR and camera labels on their own. QA figures: scored LiDAR points with the path polygon (top), and the label over the image with the 2D road outline (bottom).

In [ ]:
import json
from IPython.display import Image, display

fmt = lambda v: "n/a" if v is None else f"{v:.3f}"
for seq in ready:
    manifest = json.loads((OUT_DIR / seq / "manifest.json").read_text())
    print(f"{seq}: {manifest['frames']} frames, statuses {manifest['statuses']}")
    for part, a in manifest["audit_vs_2d_semantics"].items():
        px = a["pixels"]
        print(f"  {part:7s} IoU {fmt(a['iou'])}  precision {fmt(a['precision'])}  recall {fmt(a['recall'])}  "
              f"(pixels: TP {px['tp']:,}, FP {px['fp']:,}, FN {px['fn']:,})")
    for png in sorted((OUT_DIR / seq / "qa").glob("*.png"))[:3]:
        display(Image(filename=str(png)))
sh(f'du -sh "{OUT_DIR}"/*')

## After the run

Save the output as the private dataset **`ocgf-road-labels-v1`** (Output → New Dataset). The student's training notebooks attach it; these labels are a training target only, never evaluation ground truth.